# 🛡️ EPI Finder - Avaliação de Modelo, Métricas e Diagnósticos de SST (Fase 5)

Bem-vindo à **Fase 5** do projeto **EPI Finder**!

Nesta fase, realizamos a **avaliação formal e diagnóstica** do modelo treinado com YOLOv8 para detecção de uso de capacete de segurança. A análise é executada sobre o **conjunto de teste cego (`test split`)**, garantindo que o modelo seja auditado em dados reais nunca antes vistos durante o treinamento ou ajuste de hiperparâmetros.

---

### 🎯 Objetivos Centrais desta Fase:
1. **Validação Cega no Conjunto de Teste:** Auditar as 39 imagens e 128 anotações do split `test`.
2. **Análise Estatística com Pandas:** Estruturar as métricas globais e específicas por classe (`head` vs `helmet`) em tabelas analíticas.
3. **Diagnóstico para Segurança do Trabalho (SST):**
   - **Recall da classe `head` (Sem Capacete):** Avaliar o risco de acidentes decorrente de infrações não identificadas (Falsos Negativos).
   - **Precision da classe `head`:** Avaliar a confiabilidade dos alertas e mitigar a fadiga de alarmes falsos (Falsos Positivos).
   - **Equilíbrio com F1-Score e mAP:** Mensurar a precisão média do detector em múltiplos limiares de sobreposição ($mAP@50$ e $mAP@50\text{-}95$).
4. **Análise de Convergência do Treinamento (`results.csv`):** Explorar a trajetória das funções de perda (`box_loss`, `cls_loss`, `dfl_loss`) e a evolução do aprendizado.
5. **Inspeção de Matriz de Confusão e Curvas Operacionais:** Interpretar as confusões entre fundo (*background*), cabeças desprotegidas e capacetes.
6. **Auditoria Visual e Diagnóstico de Erros (*Error Analysis*):** Inspecionar lado a lado imagens reais com gabarito (*ground truth*) versus detecções previstas, utilizando as ferramentas do módulo `src.utils`.
7. **Diretrizes para a Fase 6:** Calibrar os limiares de corte ideais (`conf_threshold`) para a inferência em tempo real e geração de relatórios de auditoria.

## 1. Configuração do Ambiente e Importação de Bibliotecas

Configuramos o caminho raiz do projeto no `sys.path`, definimos os estilos gráficos para relatórios com **Matplotlib/Seaborn** e importamos os módulos utilitários do projeto (`src.utils` e `src.evaluate`).

In [ ]:
import os
import sys
import json
import glob
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Ultralytics YOLO
import ultralytics
from ultralytics import YOLO

# Garante que a raiz do projeto esteja no sys.path e seja o diretório de trabalho atual
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# Módulos customizados do EPI Finder
from src.utils import (
    DEFAULT_CLASSES,
    CLASS_COLORS_BGR,
    load_yolo_annotation,
    yolo_to_xyxy,
    draw_bounding_boxes,
    compute_iou,
    bgr_to_rgb
)
from src.evaluate import evaluate_yolo, calculate_f1

# Configurações visuais elegantes para os gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print(f"📁 Diretório raiz do projeto : {PROJECT_ROOT}")
print(f"🐍 Python                    : {sys.version.split()[0]}")
print(f"📊 Pandas versão             : {pd.__version__}")
print(f"🚀 Ultralytics versão        : {ultralytics.__version__}")
print("✅ Módulos 'src.utils' e 'src.evaluate' carregados com sucesso!")

## 2. Carregamento do Modelo Treinado e Metadados de Auditoria

O diretório `models/` centraliza os pesos salvos do treinamento (`best.pt`) e o arquivo `metadata.json`, que registra os parâmetros e hiperparâmetros configurados.

Aqui verificamos o estado dos pesos e carregamos o modelo para avaliação.

In [ ]:
weights_path = PROJECT_ROOT / "models" / "best.pt"
yaml_path = PROJECT_ROOT / "data" / "data.yaml"
meta_path = PROJECT_ROOT / "models" / "metadata.json"

assert weights_path.exists(), f"Pesos não encontrados em {weights_path}!"
assert yaml_path.exists(), f"Configuração data.yaml não encontrada em {yaml_path}!"

# Leitura e exibição dos metadados existentes
if meta_path.exists():
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print("📋 Metadados do Modelo Treinado (models/metadata.json):")
    print(f"   • Modelo Base       : {metadata.get('base_model', 'N/A')}")
    print(f"   • Data do Treino    : {metadata.get('training_date', 'N/A')}")
    print(f"   • Épocas Treinadas  : {metadata.get('epochs_configured', 'N/A')}")
    print(f"   • Resolução (imgsz) : {metadata.get('image_size', 640)}x{metadata.get('image_size', 640)}")
    print(f"   • Batch Size        : {metadata.get('batch_size', 'N/A')}")
    print(f"   • Dispositivo       : {metadata.get('device', 'cpu')}")
    print(f"   • Classes           : {metadata.get('classes', DEFAULT_CLASSES)}")
else:
    print("ℹ️ Arquivo metadata.json não encontrado. Prosseguindo com carregamento direto dos pesos.")

# Instancia o modelo YOLO
model = YOLO(str(weights_path))
print(f"\n✅ Modelo carregado com sucesso: {weights_path.name}")
print(f"🏷️  Classes mapeadas no modelo: {model.names}")

## 3. Avaliação Formal no Split de Teste (`test split`)

Executamos a validação cega utilizando o módulo `src.evaluate.evaluate_yolo` sobre as **39 imagens de teste** (que reúnem 128 anotações reais de cabeças e capacetes).

Diferente do split de validação (utilizado durante as épocas para salvar o `best.pt`), o conjunto de teste **nunca** interferiu em nenhuma decisão do treinamento.

In [ ]:
# Dispara a avaliação via módulo modular src.evaluate
evaluation_results = evaluate_yolo(
    weights=str(weights_path),
    data_yaml=str(yaml_path),
    split="test",
    img_size=640,
    conf_threshold=0.25,
    iou_threshold=0.60,
    save_json=str(PROJECT_ROOT / "models" / "test_metrics.json"),
    save_plots=True,
    verbose=True
)

artifacts_dir = Path(evaluation_results.get("artifacts_dir", "runs/detect/val"))
print(f"\n📁 Artefatos visuais de validação disponíveis em: {artifacts_dir}")

## 4. Análise Estatística Tabular com Pandas e Métricas de SST

Com os resultados da validação em mãos, estruturamos os dados em um `pd.DataFrame` para comparar o desempenho relativo entre as duas classes:
- **`head` (Classe 0):** Pessoa sem capacete (infração / perigo).
- **`helmet` (Classe 1):** Pessoa com capacete (conforme / seguro).

### 💡 Racional de Segurança e Saúde no Trabalho (SST):
- **O Custo do Falso Negativo (Recall de `head`):**
  Se a IA deixar de detectar uma pessoa sem capacete, ela permanecerá desprotegida no canteiro de obras, expondo a empresa a multas da NR-6 e o trabalhador ao risco fatal de traumatismo. Portanto, **maximizar o Recall de `head` é a maior prioridade**.
- **O Custo do Falso Positivo (Precision de `head`):**
  Se a IA acusar que um trabalhador com capacete está sem capacete, o alarme tocará indevidamente. Em excesso, gera a "fadiga de alarmes" nos monitores.

In [ ]:
# Monta DataFrame analítico a partir do dicionário de resultados
class_metrics = evaluation_results.get("class_metrics", {})
global_metrics = evaluation_results.get("global_metrics", {})

rows = []
for c_id, data in class_metrics.items():
    rows.append({
        "Classe": f"{data['class_name']} ({c_id})",
        "Precision": data["precision"],
        "Recall": data["recall"],
        "F1-Score": data["f1_score"],
        "mAP@50": data["map50"],
        "mAP@50-95": data["map50_95"]
    })

# Adiciona linha de média geral
rows.append({
    "Classe": "MÉDIA GERAL (ALL)",
    "Precision": global_metrics.get("precision", 0.0),
    "Recall": global_metrics.get("recall", 0.0),
    "F1-Score": global_metrics.get("f1_score", 0.0),
    "mAP@50": global_metrics.get("map50", 0.0),
    "mAP@50-95": global_metrics.get("map50_95", 0.0)
})

df_metrics = pd.DataFrame(rows).set_index("Classe")

print("📊 Tabela Analítica de Métricas (Pandas DataFrame):")
display(df_metrics.style.format("{:.4f}").background_gradient(cmap="Blues", subset=["Precision", "Recall", "F1-Score", "mAP@50", "mAP@50-95"]))

# Diagnóstico de SST
print("\n🦺 Diagnóstico do Módulo de Segurança do Trabalho:")
for k, msg in evaluation_results.get("sst_audit", {}).items():
    print(f"   • {msg}")

In [ ]:
# Gráfico comparativo de barras com Matplotlib e Seaborn
fig, ax = plt.subplots(figsize=(10, 5))

metrics_cols = ["Precision", "Recall", "F1-Score", "mAP@50"]
target_rows = [idx for idx in df_metrics.index if idx in ["head (0)", "helmet (1)", "MÉDIA GERAL (ALL)"]]
df_plot = df_metrics.loc[target_rows, metrics_cols]

df_plot.T.plot(kind="bar", ax=ax, width=0.7, colormap="viridis")
ax.set_title("Comparativo de Métricas por Classe e Média Geral (Split: Test)", pad=15)
ax.set_ylabel("Score [0.0 - 1.0]")
ax.set_xlabel("Métrica")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.7)
plt.xticks(rotation=0)
plt.legend(title="Segmento", frameon=True)
plt.tight_layout()
plt.show()

## 5. Análise da Evolução do Treinamento com Pandas (`results.csv`)

Durante o treinamento, o YOLOv8 gera a cada época um registro tabular minucioso em `runs/detect/<experimento>/results.csv`.
Abaixo, importamos e analisamos esse arquivo histórico com **Pandas** para auditar:
1. **Comportamento das perdas de treino vs validação:** `box_loss` (geometria da caixa), `cls_loss` (classificação) e `dfl_loss`.
2. **Evolução de acurácia:** $mAP@50$ e $mAP@50\text{-}95$ ao longo das épocas.
3. **Detecção de Overfitting / Ponto de Parada Ideal:** Momento exato em que a perda de validação atinge o valor mínimo e o $mAP$ atinge o ápice.

In [ ]:
# Busca automática pelo arquivo results.csv de treinamentos existentes
possible_results = sorted(glob.glob(str(PROJECT_ROOT / "runs" / "detect" / "**" / "results.csv"), recursive=True))

if possible_results:
    results_csv_path = Path(possible_results[-1])
    print(f"📊 Carregando histórico de épocas a partir de: {results_csv_path}")
    df_history = pd.read_csv(results_csv_path)
    df_history.columns = [c.strip() for c in df_history.columns]
    
    print(f"✅ Total de épocas registradas: {len(df_history)}")
    display(df_history.head(5))
    
    # Identificação da melhor época por mAP50
    map50_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df_history.columns else "metrics/mAP50"
    if map50_col in df_history.columns and not df_history.empty:
        best_epoch_idx = df_history[map50_col].idxmax()
        best_epoch = df_history.loc[best_epoch_idx, "epoch"]
        best_map = df_history.loc[best_epoch_idx, map50_col]
        print(f"\n🏆 Melhor Época Registrada: Época {int(best_epoch)} com mAP@50 = {best_map:.4f}")
        
        # Plot das Curvas de Perdas e Métricas
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 1. Gráfico de Perdas (Losses)
        if "train/box_loss" in df_history.columns and "val/box_loss" in df_history.columns:
            axes[0].plot(df_history["epoch"], df_history["train/box_loss"], label="Train Box Loss", color="#e74c3c")
            axes[0].plot(df_history["epoch"], df_history["val/box_loss"], label="Val Box Loss", linestyle="--", color="#c0392b")
        if "train/cls_loss" in df_history.columns and "val/cls_loss" in df_history.columns:
            axes[0].plot(df_history["epoch"], df_history["train/cls_loss"], label="Train Cls Loss", color="#3498db")
            axes[0].plot(df_history["epoch"], df_history["val/cls_loss"], label="Val Cls Loss", linestyle="--", color="#2980b9")
        axes[0].set_title("Evolução das Funções de Perda (Losses)")
        axes[0].set_xlabel("Época")
        axes[0].set_ylabel("Perda")
        axes[0].legend()
        axes[0].grid(True, linestyle="--", alpha=0.7)
        
        # 2. Gráfico de mAP
        axes[1].plot(df_history["epoch"], df_history[map50_col], label="Val mAP@50", color="#2ecc71", linewidth=2)
        map50_95_col = "metrics/mAP50-95(B)" if "metrics/mAP50-95(B)" in df_history.columns else "metrics/mAP50-95"
        if map50_95_col in df_history.columns:
            axes[1].plot(df_history["epoch"], df_history[map50_95_col], label="Val mAP@50-95", color="#27ae60", linestyle="--")
        axes[1].axvline(x=best_epoch, color="#f39c12", linestyle=":", label=f"Melhor Época ({int(best_epoch)})")
        axes[1].set_title("Evolução das Métricas de Acurácia (mAP)")
        axes[1].set_xlabel("Época")
        axes[1].set_ylabel("mAP")
        axes[1].legend()
        axes[1].grid(True, linestyle="--", alpha=0.7)
        
        plt.tight_layout()
        plt.show()
else:
    print("ℹ️ Nenhum arquivo 'results.csv' de treino anterior foi localizado em 'runs/detect/'.")
    print("   Isso é normal se você estiver iniciando a avaliação antes do ciclo completo de treinamento da Fase 4.")

## 6. Inspeção de Matrizes de Confusão e Curvas Operacionais

A Ultralytics salva automaticamente diagramas de diagnóstico essenciais na pasta de validação:
1. **Matriz de Confusão Normalizada:** Mostra o percentual de acerto de cada classe e o percentual de confusão com o **Background** (fundo / objetos não anotados).
2. **Curva Precision-Recall (PR):** Ilustra a relação de troca entre precisão e sensibilidade para cada classe.
3. **Curva F1-Confidence:** Aponta em qual limiar de confiança o $F1\text{-score}$ atinge seu pico máximo.

In [ ]:
plots_to_inspect = [
    ("Matriz de Confusão Normalizada", artifacts_dir / "confusion_matrix_normalized.png"),
    ("Matriz de Confusão (Contagens Absolutas)", artifacts_dir / "confusion_matrix.png"),
    ("Curva Precision-Recall (BoxPR)", artifacts_dir / "BoxPR_curve.png"),
    ("Curva F1-Confidence (BoxF1)", artifacts_dir / "BoxF1_curve.png"),
    ("Curva Precision-Confidence (BoxP)", artifacts_dir / "BoxP_curve.png"),
    ("Curva Recall-Confidence (BoxR)", artifacts_dir / "BoxR_curve.png")
]

for title, img_path in plots_to_inspect:
    if img_path.exists():
        print(f"\n🖼️  Exibindo: {title} ({img_path.name})")
        img = Image.open(img_path)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(title, fontsize=14, pad=10)
        plt.show()
    else:
        print(f"ℹ️ Gráfico '{img_path.name}' não disponível em {artifacts_dir}.")

## 7. Auditoria Visual e Diagnóstico de Erros (Error Analysis)

Apenas métricas numéricas não revelam *onde* ou *por que* o modelo comete equívocos.
Aqui, realizamos uma **Auditoria Visual Lado a Lado**:
- **Painel da Esquerda:** Gabarito Real (*Ground Truth* das anotações manuais).
- **Painel da Direita:** Predição gerada pelo modelo treinado.

Utilizamos as funções matemáticas e de renderização de `src.utils` (`load_yolo_annotation`, `yolo_to_xyxy`, `draw_bounding_boxes`, `compute_iou`) para auditar a sobreposição geométrica e a classificação.

In [ ]:
test_img_dir = PROJECT_ROOT / "data" / "dataset" / "test" / "images"
test_lbl_dir = PROJECT_ROOT / "data" / "dataset" / "test" / "labels"

test_image_paths = sorted(glob.glob(str(test_img_dir / "*.jpg")))
print(f"🔍 Total de imagens no split de teste: {len(test_image_paths)}")

def audit_sample_image(img_path_str: str, model_instance: YOLO, conf: float = 0.25):
    """
    Renderiza lado a lado o Ground Truth e as Predições do Modelo YOLOv8.
    """
    img_path = Path(img_path_str)
    label_path = test_lbl_dir / f"{img_path.stem}.txt"
    
    # 1. Carrega imagem original com OpenCV
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        print(f"Erro ao carregar imagem: {img_path}")
        return
    h, w = img_bgr.shape[:2]
    
    # 2. Carrega e desenha o Ground Truth
    gt_boxes = []
    if label_path.exists():
        annotations = load_yolo_annotation(str(label_path))
        for ann in annotations:
            cls_id = int(ann["class_id"])
            x1, y1, x2, y2 = yolo_to_xyxy(ann["x_center"], ann["y_center"], ann["width"], ann["height"], w, h)
            gt_boxes.append({"box": [x1, y1, x2, y2], "class_id": cls_id, "confidence": None})
            
    gt_rendered_bgr = draw_bounding_boxes(
        img_bgr,
        gt_boxes,
        class_names=DEFAULT_CLASSES,
        colors=CLASS_COLORS_BGR,
        thickness=2
    )
    gt_rendered_rgb = bgr_to_rgb(gt_rendered_bgr)
    
    # 3. Executa a inferência com o modelo
    pred_results = model_instance.predict(str(img_path), conf=conf, verbose=False)
    pred_boxes = []
    for r in pred_results:
        for box in r.boxes:
            b = box.xyxy[0].cpu().numpy().astype(int)
            c = int(box.cls[0].cpu().numpy())
            cf = float(box.conf[0].cpu().numpy())
            pred_boxes.append({"box": [b[0], b[1], b[2], b[3]], "class_id": c, "confidence": cf})
            
    pred_rendered_bgr = draw_bounding_boxes(
        img_bgr,
        pred_boxes,
        class_names=DEFAULT_CLASSES,
        colors=CLASS_COLORS_BGR,
        thickness=2
    )
    pred_rendered_rgb = bgr_to_rgb(pred_rendered_bgr)
    
    # 4. Plota lado a lado
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(gt_rendered_rgb)
    axes[0].set_title(f"Gabarito Real (Ground Truth) - {len(gt_boxes)} caixas\n{img_path.name}", pad=8)
    axes[0].axis("off")
    
    axes[1].imshow(pred_rendered_rgb)
    axes[1].set_title(f"Predição do Modelo (YOLOv8) - {len(pred_boxes)} caixas (Conf >= {conf})\n{img_path.name}", pad=8)
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    # Imprime tabela com predições vs ground truth
    print(f"📋 Diagnóstico de Amostra: {img_path.name}")
    print(f"   • Anotações Reais : {[DEFAULT_CLASSES.get(b['class_id'], b['class_id']) for b in gt_boxes]}")
    print(f"   • Predições da IA : {[f'{DEFAULT_CLASSES.get(b["class_id"], b["class_id"])} ({b["confidence"]:.2%})' for b in pred_boxes]}")
    print("-" * 75)

# Audita as primeiras 3 amostras do conjunto de teste
for sample_path in test_image_paths[:3]:
    audit_sample_image(sample_path, model, conf=0.25)

## 8. Conclusões, Calibração de Confiança e Recomendações para a Fase 6

A partir dos diagnósticos levantados nesta **Fase 5**, definimos as diretrizes operacionais para a **Fase 6: Aplicação de Inferência e Relatórios**:

### 🎯 Diretrizes de Calibração para Produção:

| Parâmetro | Recomendação | Justificativa de Engenharia e SST |
|---|---|---|
| **Limiar de Confiança (`conf`)** | `0.25` a `0.35` | Equilibra a sensibilidade (Recall) para captura de infrações com a supressão de ruídos de fundo. |
| **Limiar de IoU (NMS)** | `0.50` a `0.60` | Agrupa caixas redundantes sobre a mesma cabeça sem suprimir trabalhadores que estejam ombro a ombro. |
| **Paleta Visual de Cores** | Vermelho (`head`) / Verde (`helmet`) | Facilita a identificação imediata por operadores humanos em monitores de segurança. |
| **Frequência de Relatórios** | Consolidação em `.csv` | Permite rastrear histórico de infrações com data, hora, câmera e nível de confiança para o SESMT. |

---

**🎉 Fim da Fase 5! O pipeline de avaliação está consolidado e pronto para a Fase 6 (Inferência em Vídeo e Relatórios de Conformidade).**